# HachimiMT — Dịch Trung → Việt (Google Colab)

Chạy app dịch truyện **HachimiMT / MoxhiMT** (CTranslate2) ngay trên Colab — tận dụng **CPU/GPU miễn phí** của Google, không cần cài gì trên máy.

> ## ⚡ BẬT GPU TRƯỚC (rất nên) — nhanh gấp NHIỀU LẦN
> Menu **`Runtime → Change runtime type → T4 GPU`** → `Save`, **rồi** mới Run all.
> (Colab tiếng Việt: **`Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU → Lưu`**.)
>
> ⚠️ **Chọn đúng `T4 GPU`** — KHÔNG chọn **TPU** (v5e-1...). CTranslate2 chỉ chạy GPU NVIDIA/CPU, **không dùng được TPU**: chọn TPU thì app rớt về CPU (chậm), phí tài nguyên.
>
> App tự nhận GPU và notebook đã tối ưu window cho T4. **GPU T4 dịch ~40.000 chữ Hán/giây ở beam 1** hoặc **~28.000 chữ/giây ở beam 2** (đo thật) — cả bộ truyện dài **2,4 triệu chữ khoảng 1–1,5 phút**. CPU chậm hơn nhiều lần. GPU miễn phí, bật là dùng.

**Cách dùng:** sau khi bật GPU, `Runtime → Run all` (Ctrl+F9) — tiếng Việt: `Thời gian chạy → Chạy tất cả`. Cell cuối in ra một **link công khai** (`*.gradio.live`) — bấm vào đó để mở giao diện dịch.

Mã nguồn: tải từ HF Space [ngocdang83/HachimiMT-demo](https://huggingface.co/spaces/ngocdang83/HachimiMT-demo).

### 1. Tải mã nguồn app (từ HF Space, ~60 KB)

In [1]:
import urllib.request, zipfile, os, shutil

ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
# Xóa bản giải nén cũ trước (nếu chạy lại) — tránh file/module cũ còn sót
# khi zip mới đã bỏ bớt, dẫn tới chạy code lai.
shutil.rmtree("hachimimt", ignore_errors=True)
with zipfile.ZipFile("hachimimt-local.zip") as z:
    z.extractall(".")
print("Đã tải + giải nén:", sorted(os.listdir("hachimimt")))

Đã tải + giải nén: ['.gitignore', 'HachimiMT_Benchmark_2xT4.ipynb', 'HachimiMT_Benchmark_CPU.ipynb', 'HachimiMT_Benchmark_Profile.ipynb', 'HachimiMT_Benchmark_T4.ipynb', 'HachimiMT_Benchmark_app_2xT4.ipynb', 'HachimiMT_Colab.ipynb', 'HachimiMT_Diag_speed.ipynb', 'HachimiMT_Kaggle.ipynb', 'README.md', 'README_SPACE.md', 'README_hachimimt_colab.md', 'docs', 'requirements-pytorch.txt', 'requirements.txt', 'setup.bat', 'setup_macos.sh', 'src', 'start.bat', 'start_macos.sh', 'stop.bat', 'stop_macos.sh']


### 2. Cài thư viện (CTranslate2 + Gradio + tokenizer) — ~1 phút

> ⚠️ Nếu thấy dòng đỏ `ERROR: pip's dependency resolver ... google-adk ... starlette ... incompatible` — **đây là cảnh báo vô hại**, do gói `google-adk` Colab cài sẵn (app không dùng tới). Thư viện app vẫn cài xong, **cứ chạy tiếp cell dưới**.

In [2]:
!pip install -q -r hachimimt/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 29.2 MB/s eta 0:00:00


### 3. Chạy app — đợi link `*.gradio.live` hiện ra rồi bấm vào

Lần dịch đầu sẽ tự tải model (~57 MB) từ Hugging Face. Để dừng: bấm nút ⏹ stop của cell.

In [3]:
import os, sys

# Bật link share công khai (gradio.live) — app đọc cờ này.
os.environ["HACHIMIMT_SHARE"] = "1"
# Tối ưu cho Colab T4 x1: benchmark thực tế cho thấy window=16 nhanh hơn window=4 ~25–31%.
os.environ.setdefault("HACHIMIMT_BATCH_SIZE", "96")
os.environ.setdefault("HACHIMIMT_CT2_BATCH_TYPE", "tokens")
os.environ.setdefault("HACHIMIMT_CT2_WINDOW_MULTIPLIER", "16")
os.environ.setdefault("HACHIMIMT_INTER_THREADS", "1")
sys.path.insert(0, os.path.abspath("hachimimt/src"))

# Báo rõ đang chạy GPU hay CPU (để biết đã bật GPU chưa).
from hardware import detect_hardware_profile
_hw = detect_hardware_profile()
if _hw.has_cuda:
    print(f"✅ Đang dùng GPU: {_hw.gpu_name} — nhanh!  ({_hw.summary})")
    print("⚙️ Colab speed config: batch=96, batch_type=tokens, window=16, inter=1")
else:
    print("⚠️ Đang chạy CPU (chậm ~50× GPU). Bật GPU: Runtime → Change runtime type → "
          "T4 GPU → Save (tiếng Việt: Thời gian chạy → Thay đổi loại thời gian chạy → "
          "T4 GPU → Lưu), rồi Run all lại.\n"
          "   Lưu ý: chọn T4 GPU, KHÔNG chọn TPU — app không dùng được TPU, sẽ rớt về CPU.")

# Chạy app.
import app
app.main()

⚠️ Đang chạy CPU (chậm ~50× GPU). Bật GPU: Runtime → Change runtime type → T4 GPU → Save (tiếng Việt: Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU → Lưu), rồi Run all lại.
   Lưu ý: chọn T4 GPU, KHÔNG chọn TPU — app không dùng được TPU, sẽ rớt về CPU.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://257946bf8620a71952.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
